## checking for empty station/files

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------------
# Path to combined AMSR-LPRM–in-situ surface-layer files
# ------------------------------------------------------------------
COMBINE_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine"
)


# ------------------------------------------------------------------
# Containers for reports
# ------------------------------------------------------------------
missing_file = []        # (network, station)
missing_cols = []        # (network, station, [cols])
empty_overlap = []       # (network, station)

all_nan_ts_station_k = []   # (network, station)
all_nan_ts_lprm = []        # (network, station)
no_valid_pairs = []         # (network, station)


# Required columns in combined files (LPRM version)
required_cols = [
    "date", "time",
    "ts_station_k",   # in-situ (K)
    "ts_lprm",        # AMSR2 LPRM (K)
    "lat", "lon",
    "lat_lprm", "lon_lprm",
    "elev", "cc", "lc",
    "land_cover_group", "climate_group",
    "temp_class", "elevation_class",
]


# ------------------------------------------------------------------
# Scan all networks and stations under COMBINE_ROOT
# ------------------------------------------------------------------
if not COMBINE_ROOT.exists():
    raise FileNotFoundError(f"COMBINE_ROOT does not exist: {COMBINE_ROOT}")


for network_dir in sorted(COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        st = station_dir.name

        # Look for combined insitu+LPRM file(s)
        csv_list = list(station_dir.glob("*_insitu_lprm_surface_temperature.csv"))
        if len(csv_list) == 0:
            missing_file.append((net, st))
            continue

        csv_path = csv_list[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception:
            missing_file.append((net, st))
            continue

        # Check required columns
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            missing_cols.append((net, st, missing))
            continue

        # Check for any data rows
        if df.empty:
            empty_overlap.append((net, st))
            continue

        # ------------------------------------------------------------------
        # Validity checks for ts_station_k and ts_lprm
        # ------------------------------------------------------------------
        if df["ts_station_k"].isna().all():
            all_nan_ts_station_k.append((net, st))

        if df["ts_lprm"].isna().all():
            all_nan_ts_lprm.append((net, st))

        # Check whether there is at least one timestamp
        # with both values present (valid pair)
        both_ok = (~df["ts_station_k"].isna()) & (~df["ts_lprm"].isna())
        if not both_ok.any():
            no_valid_pairs.append((net, st))


# ------------------------------------------------------------------
# Print summary
# ------------------------------------------------------------------
print("\n=== Stations with NO combined file ===")
if not missing_file:
    print("  None")
else:
    for net, st in sorted(missing_file):
        print(f"  {net} / {st}")

print("\n=== Stations with combined file but MISSING columns ===")
if not missing_cols:
    print("  None")
else:
    for net, st, miss in missing_cols:
        print(f"  {net} / {st} -> missing: {', '.join(miss)}")

print("\n=== Stations with empty combined file (no rows) ===")
if not empty_overlap:
    print("  None")
else:
    for net, st in empty_overlap:
        print(f"  {net} / {st}")

print("\n=== Stations with INVALID ts_station_k (all NaN) ===")
if not all_nan_ts_station_k:
    print("  None")
else:
    for net, st in all_nan_ts_station_k:
        print(f"  {net} / {st}")

print("\n=== Stations with INVALID ts_lprm (all NaN) ===")
if not all_nan_ts_lprm:
    print("  None")
else:
    for net, st in all_nan_ts_lprm:
        print(f"  {net} / {st}")

print("\n=== Stations with NO valid ts_station_k–ts_lprm pairs ===")
if not no_valid_pairs:
    print("  None")
else:
    for net, st in no_valid_pairs:
        print(f"  {net} / {st}")



=== Stations with NO combined file ===
  ARM / Anthony
  ARM / Ashton
  ARM / Byron
  ARM / Lamont-CF1
  ARM / MapleCity
  ARM / Marshall
  ARM / Medford
  ARM / Morrison
  ARM / Newkirk
  ARM / Okmulgee
  ARM / Omega
  ARM / Pawhuska
  ARM / Ringwood
  ARM / Tryon
  ARM / Tyro
  ARM / Waukomis
  BIEBRZA_S-1 / grassland-soil-1
  BIEBRZA_S-1 / grassland-soil-2
  BIEBRZA_S-1 / grassland-soil-3
  BIEBRZA_S-1 / grassland-soil-4
  BIEBRZA_S-1 / grassland-soil-5
  BIEBRZA_S-1 / grassland-soil-6
  BIEBRZA_S-1 / grassland-soil-7
  BIEBRZA_S-1 / grassland-soil-8
  BIEBRZA_S-1 / grassland-soil-9
  BIEBRZA_S-1 / marshland-soil-11
  BIEBRZA_S-1 / marshland-soil-12
  BIEBRZA_S-1 / marshland-soil-13
  BIEBRZA_S-1 / marshland-soil-14
  BIEBRZA_S-1 / marshland-soil-15
  BIEBRZA_S-1 / marshland-soil-16
  BIEBRZA_S-1 / marshland-soil-17
  BIEBRZA_S-1 / marshland-soil-18
  BIEBRZA_S-1 / marshland-soil-19
  BNZ-LTER / FP1A
  BNZ-LTER / FP2A
  BNZ-LTER / FP3A
  BNZ-LTER / FP4A
  BNZ-LTER / FP5A
  BNZ-LTER

In [2]:
from pathlib import Path
import pandas as pd

# Root where all combined ERA5–in situ files live
ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine")

n_files_scanned = 0
n_files_renamed = 0

for network_dir in sorted(ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        st = station_dir.name

        # Rename in every CSV in this station directory
        for csv_path in station_dir.glob("*.csv"):
            n_files_scanned += 1
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"[WARN] Could not read {csv_path}: {e}")
                continue

            if "ts_station" not in df.columns:
                # nothing to change here
                continue

            # Avoid accidental overwrite if ts_lprm already exists
            if "ts_lprm" in df.columns:
                print(f"[WARN] {csv_path} already has 'ts_lprm', skipping.")
                continue

            df = df.rename(columns={"ts_station": "ts_lprm"})
            df.to_csv(csv_path, index=False, float_format="%.3f")
            n_files_renamed += 1
            print(f"Renamed ts_station -> ts_lprm in {csv_path}")

print(f"\nTotal CSV files scanned: {n_files_scanned}")
print(f"Files updated (renamed): {n_files_renamed}")



Total CSV files scanned: 1362
Files updated (renamed): 0


## Computing the SD and correlation

In [3]:
import pandas as pd
from pathlib import Path


# ----------------------------------------------------------------------
# Root dirs (AMSR2 LPRM – in situ combine)
# ----------------------------------------------------------------------
LPRM_COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine")

OUT_SUMMARY = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv"
)


results = []

# ----------------------------------------------------------------------
# Loop over networks and stations
# ----------------------------------------------------------------------
for network_dir in sorted(LPRM_COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    network = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        station = station_dir.name

        # One combined file per station (as produced by your combine script)
        csv_files = list(station_dir.glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV in {station_dir}, skipping.")
            continue

        csv_path = csv_files[0]

        # ------------------------------------------------------------------
        # Read CSV
        # ------------------------------------------------------------------
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[ERROR] Cannot read {csv_path}: {e}")
            results.append({
                "network": network,
                "station": station,
                "n_points": 0,
                "sd_insitu": None,
                "sd_lprm": None,
                "sd_ratio_lprm_insitu": None,
                "corr_lprm_insitu": None,
                "pass_sd_filter": False,
                "pass_corr_filter": False,
            })
            continue

        # Optional: build datetime (not used in SD/corr, but harmless)
        if "date" in df.columns and "time" in df.columns:
            df["datetime"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce"
            )

        # ------------------------------------------------------------------
        # Check required columns
        # ------------------------------------------------------------------
        if not {"ts_station_k", "ts_lprm"}.issubset(df.columns):
            print(f"[WARN] Missing ts_station_k or ts_lprm in {csv_path}, skipping station.")
            continue

        # ------------------------------------------------------------------
        # Drop rows with NaNs in either series (you already enforced paired NaNs)
        # ------------------------------------------------------------------
        sub = df[["ts_station_k", "ts_lprm"]].dropna()
        n_points = len(sub)

        # ------------------------------------------------------------------
        # Compute SDs, SD ratio, and correlation
        # ------------------------------------------------------------------
        if n_points < 2:
            print(f"[WARN] Not enough valid points for {network}/{station} ({n_points}), skipping stats.")
            sd_insitu = sd_lprm = ratio = corr = None
            pass_sd = pass_corr = False
        else:
            sd_insitu = sub["ts_station_k"].std()
            sd_lprm = sub["ts_lprm"].std()

            if sd_insitu is not None and sd_insitu > 0:
                ratio = sd_lprm / sd_insitu
            else:
                ratio = None

            corr = sub["ts_station_k"].corr(sub["ts_lprm"])

            # SD filter: same thresholds as ERA5 (0.1 <= ratio <= 3)
            if ratio is not None and 0.1 <= ratio <= 3:
                pass_sd = True
            else:
                pass_sd = False

            # Correlation filter: same threshold as ERA5 (corr >= 0.5)
            pass_corr = corr is not None and corr >= 0.5

        # ------------------------------------------------------------------
        # Collect station result
        # ------------------------------------------------------------------
        results.append({
            "network": network,
            "station": station,
            "n_points": n_points,
            "sd_insitu": sd_insitu,
            "sd_lprm": sd_lprm,
            "sd_ratio_lprm_insitu": ratio,
            "corr_lprm_insitu": corr,
            "pass_sd_filter": pass_sd,
            "pass_corr_filter": pass_corr,
        })

# ----------------------------------------------------------------------
# Build DataFrame and save summary
# ----------------------------------------------------------------------
summary_df = pd.DataFrame(results)

# Round numeric columns to 3 decimals
num_cols = ["sd_insitu", "sd_lprm", "sd_ratio_lprm_insitu", "corr_lprm_insitu"]
for col in num_cols:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].round(3)

OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved summary to {OUT_SUMMARY}")


/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

[WARN] Not enough valid points for COSMOS-UK/CocklePark (0), skipping stats.
[WARN] Not enough valid points for COSMOS-UK/CwmGarw (0), skipping stats.


/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

[WARN] Not enough valid points for COSMOS-UK/HenfaesFarm (0), skipping stats.
[WARN] Not enough valid points for COSMOS-UK/HollinHill (0), skipping stats.
[WARN] Not enough valid points for COSMOS-UK/LullingtonHeath (0), skipping stats.


/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(


...

/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

[WARN] Not enough valid points for WSMN/WSMN-1 (0), skipping stats.
[WARN] Not enough valid points for WSMN/WSMN-2 (0), skipping stats.
[WARN] Not enough valid points for WSMN/WSMN-3 (0), skipping stats.
[WARN] Not enough valid points for WSMN/WSMN-4 (0), skipping stats.
[WARN] Not enough valid points for WSMN/WSMN-5 (0), skipping stats.
[WARN] Not enough valid points for WSMN/WSMN-6 (0), skipping stats.


/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

Saved summary to /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv


/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(
/tmp/ipykernel_104525/1249083415.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(


## Checking the failed files

In [4]:
import pandas as pd
from pathlib import Path

summary_path = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv")

df = pd.read_csv(summary_path)

# Total stations
total = len(df)

# Stations that pass both SD and correlation filters
pass_both = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]

# Stations failing at least one filter
fail_any = df[(df["pass_sd_filter"] == False) | (df["pass_corr_filter"] == False)]

print(f"Total stations          : {total}")
print(f"Pass both (SD & corr)   : {len(pass_both)}")
print(f"Fail SD or corr (or both): {len(fail_any)}")

# Optional: counts per network
print("\nPer network (pass both):")
print(pass_both.groupby("network")["station"].count())

print("\nPer network (fail any):")
print(fail_any.groupby("network")["station"].count())

# NEW: print failing station names per network
print("\nFailing stations by network:")
for network, sub in fail_any.groupby("network"):
    station_list = sorted(sub["station"].unique())
    print(f"{network}: {station_list} = {len(station_list)}")


Total stations          : 1362
Pass both (SD & corr)   : 1195
Fail SD or corr (or both): 167

Per network (pass both):
network
ARM                    16
BIEBRZA_S-1            18
BNZ-LTER               10
Berlin                  1
COSMOS-UK              42
CTP_SMTMN              57
CW3E                    7
DAHRA                   1
FLUXNET-AMERIFLUX       7
FMI                    25
FR_Aqui                 5
HOAL                   29
HOBE                   30
HYDROL-NET_PERUGIA      1
IMA_CAN1               12
IPE                     1
LAB-net                 3
LABFLUX                 1
MAQU                   21
MOL-RAO                 2
NAQU                   10
OZNET                  17
REMEDHUS               20
RISMA                  23
ROMPS                   4
RSMN                   20
SCAN                  191
SKKU                    1
SMN-SDR                33
SMOSMANIA              20
SNOTEL                355
STEMS                   4
SWEX_POLAND             1
TAHMO          

## Pixel mean

In [13]:
import os
import glob
import pandas as pd
from collections import Counter
from pathlib import Path


# ---------------------------------------------------------------------
# Paths for AMSR2–LPRM surface layer
# ---------------------------------------------------------------------


COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine"
OUT_ROOT     = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/pixel_mean"

ERA5_SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv"
LPRM_SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv"


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------


def majority(series):
    """Most common non-NaN value in a series."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return Counter(vals).most_common(1)[0][0]


def classify_elevation(elev):
    """Classify elevation using DEM-I/II/III/IV thresholds."""
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"

    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"


def load_pass_stations(era5_summary_path, lprm_summary_path):
    """
    Read ERA5 and LPRM summary CSVs and return a set of (network, station)
    pairs that are considered GOOD: they do NOT fail both products.

    Logic:
    - A station passes ERA5 if pass_sd_filter & pass_corr_filter are True in ERA5 summary.
    - A station passes LPRM if pass_sd_filter & pass_corr_filter are True in LPRM summary.
    - A station is discarded only if it fails BOTH.
    - So we keep all stations that pass at least one product.
    """
    df_era5 = pd.read_csv(era5_summary_path)
    df_lprm = pd.read_csv(lprm_summary_path)

    ok_era5 = df_era5[(df_era5["pass_sd_filter"] == True) & (df_era5["pass_corr_filter"] == True)]
    ok_lprm = df_lprm[(df_lprm["pass_sd_filter"] == True) & (df_lprm["pass_corr_filter"] == True)]

    set_era5 = set(zip(ok_era5["network"], ok_era5["station"]))
    set_lprm = set(zip(ok_lprm["network"], ok_lprm["station"]))

    # Good stations are those that pass at least one product
    pass_any = set_era5.union(set_lprm)
    return pass_any


# ---------------------------------------------------------------------
# Standard processing for moderate-size networks
# ---------------------------------------------------------------------


def process_network(network, pass_stations):
    base_in    = os.path.join(COMBINE_ROOT, network)
    base_out   = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network: {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    all_records = []
    used_station_dirs = []

    # Load station data, but only for stations that passed filters
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            print(f"  Skipping {network}/{station_name} (failed in both ERA5 & LPRM summaries).")
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            # Required columns for LPRM surface layer
            required_cols = {
                "date", "time",
                "ts_station_k", "ts_lprm",
                "lat", "lon", "elev",
                "cc", "lc",
                "land_cover_group", "climate_group", "temp_class",
                "elevation_class",
                "lat_lprm", "lon_lprm",
            }
            if not required_cols.issubset(df.columns):
                print(f"  [WARN] Missing required columns in {f}, skipping this file.")
                continue

            df["station"] = station_name
            all_records.append(df)
            used_station_dirs.append(station_dir)

    if not all_records:
        print(f"No valid CSV files found under {base_in}/* (after filtering), skipping.")
        return

    df_all = pd.concat(all_records, ignore_index=True)
    print(f"Loaded {len(set(used_station_dirs))} stations after filtering, total rows: {len(df_all)}")

    # DENSE PIXELS: at least 2 stations in same (lat_lprm, lon_lprm)
    pixel_station_counts = (
        df_all.groupby(["lat_lprm", "lon_lprm"])["station"]
              .nunique()
              .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_lprm", "lon_lprm"]]
    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    n_sparse_files = 0
    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # DENSE
    print("Processing dense pixels (pixel-mean files)...")
    df_dense_all = df_all.merge(dense_pixels, on=["lat_lprm", "lon_lprm"], how="inner")
    print("Rows in dense pixels (before grouping):", len(df_dense_all))

    stations_in_dense = set()

    # Also read existing dense files to populate stations_in_dense (idempotent)
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    if not df_dense_all.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_lprm"]
            plon = pix["lon_lprm"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            # Skip this pixel if its dense file already exists
            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_lprm={plat}, lon_lprm={plon} (file exists).")
                continue

            df_pix = df_dense_all[
                (df_dense_all["lat_lprm"] == plat) &
                (df_dense_all["lon_lprm"] == plon)
            ].copy()

            if df_pix.empty:
                continue

            print(f"  Dense pixel at lat_lprm={plat}, lon_lprm={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref  = g_valid["ts_station_k"].mean()
                    ts_lprm = g_valid["ts_lprm"].iloc[0]

                    lat_val  = majority(g_valid["lat"])
                    lon_val  = majority(g_valid["lon"])
                    cc_val   = majority(g_valid["cc"])
                    lc_val   = majority(g_valid["lc"])
                    lcg_val  = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref  = pd.NA
                    ts_lprm = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_lprm": ts_lprm,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            # If no rows produced for this pixel (no times with >=2 valid stations), skip it
            if not out_rows:
                print(f"  No valid time steps (>=2 stations) for dense pixel lat_lprm={plat}, lon_lprm={plon}, skipping.")
                continue

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_lprm"]      = pd.to_numeric(df_pix_mean["ts_lprm"],      errors="coerce").round(3)

            df_pix_mean["lat_lprm"] = plat
            df_pix_mean["lon_lprm"] = plon
            df_pix_mean["stations"] = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")
    else:
        print("No dense pixels found to process.")

    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")

    # SPARSE: station-wise files, excluding stations used in any dense pixel
    print("Writing sparse (station-wise) files (excluding dense stations)...")
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Stations found (raw): {n_stations_found}")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


# ---------------------------------------------------------------------
# SNOTEL-specific processing (memory-safe)
# ---------------------------------------------------------------------


def process_network_snotel(pass_stations):
    network = "SNOTEL"
    base_in    = os.path.join(COMBINE_ROOT, network)
    base_out   = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network (chunked): {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    # 1) FIRST PASS: determine dense pixels using minimal columns
    pixel_station_pairs = []

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f, usecols=["lat_lprm", "lon_lprm"])
            except Exception as e:
                print(f"  [WARN] Could not read minimal cols from {f}: {e}")
                continue

            pix_unique = df.dropna(subset=["lat_lprm", "lon_lprm"]).drop_duplicates()
            for _, row in pix_unique.iterrows():
                pixel_station_pairs.append(
                    (row["lat_lprm"], row["lon_lprm"], station_name)
                )

    if not pixel_station_pairs:
        print("No pixel/station info found for SNOTEL, skipping.")
        return

    df_pix_st = pd.DataFrame(
        pixel_station_pairs,
        columns=["lat_lprm", "lon_lprm", "station"]
    )

    pixel_station_counts = (
        df_pix_st.groupby(["lat_lprm", "lon_lprm"])["station"]
        .nunique()
        .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_lprm", "lon_lprm"]]

    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    stations_in_dense = set()

    # Existing dense files
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # 2) SECOND PASS: process each dense pixel separately
    if not dense_pixels.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_lprm"]
            plon = pix["lon_lprm"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_lprm={plat}, lon_lprm={plon} (file exists).")
                continue

            # stations that ever fall into this pixel
            stations_pix = df_pix_st[
                (df_pix_st["lat_lprm"] == plat) &
                (df_pix_st["lon_lprm"] == plon)
            ]["station"].unique()

            df_pix_all = []

            for station_name in stations_pix:
                if (network, station_name) not in pass_stations:
                    continue

                station_dir = os.path.join(base_in, station_name)
                csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
                if not csv_files:
                    continue

                for f in csv_files:
                    try:
                        df = pd.read_csv(f)
                    except Exception as e:
                        print(f"  [WARN] Could not read {f}: {e}")
                        continue

                    required_cols = {
                        "date", "time", "ts_station_k", "ts_lprm",
                        "lat_lprm", "lon_lprm",
                        "lat", "lon",
                        "cc", "lc",
                        "land_cover_group", "climate_group", "temp_class", "elev"
                    }
                    if not required_cols.issubset(df.columns):
                        continue

                    df = df[
                        (df["lat_lprm"] == plat) &
                        (df["lon_lprm"] == plon)
                    ].copy()

                    if df.empty:
                        continue

                    df["station"] = station_name
                    df_pix_all.append(df)

            if not df_pix_all:
                continue

            df_pix = pd.concat(df_pix_all, ignore_index=True)

            print(f"  Dense pixel at lat_lprm={plat}, lon_lprm={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref  = g_valid["ts_station_k"].mean()
                    ts_lprm = g_valid["ts_lprm"].iloc[0]

                    lat_val  = majority(g_valid["lat"])
                    lon_val  = majority(g_valid["lon"])
                    cc_val   = majority(g_valid["cc"])
                    lc_val   = majority(g_valid["lc"])
                    lcg_val  = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref  = pd.NA
                    ts_lprm = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_lprm": ts_lprm,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            if not out_rows:
                print(f"  No valid time steps (>=2 stations) for dense pixel lat_lprm={plat}, lon_lprm={plon}, skipping.")
                continue

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_lprm"]      = pd.to_numeric(df_pix_mean["ts_lprm"],      errors="coerce").round(3)
            df_pix_mean["lat_lprm"] = plat
            df_pix_mean["lon_lprm"] = plon
            df_pix_mean["stations"] = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")

    else:
        print("No dense pixels found for SNOTEL.")

    # 3) SPARSE: per-station outputs for stations not in dense pixels
    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")
    print("Writing sparse (station-wise) files (excluding dense stations)...")

    n_sparse_files = 0

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------


if __name__ == "__main__":
    pass_stations = load_pass_stations(ERA5_SUMMARY_PATH, LPRM_SUMMARY_PATH)

    network_dirs = sorted(
        d for d in glob.glob(os.path.join(COMBINE_ROOT, "*"))
        if os.path.isdir(d)
    )
    networks = [os.path.basename(d) for d in network_dirs]

    print("Networks found:", networks)
    for net in networks:
        if net == "SNOTEL":
            process_network_snotel(pass_stations)
        else:
            process_network(net, pass_stations)


Networks found: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'HYDROL-NET_PERUGIA', 'IMA_CAN1', 'IPE', 'LAB-net', 'LABFLUX', 'MAQU', 'MOL-RAO', 'MySMNet', 'NAQU', 'NGARI', 'OZNET', 'REMEDHUS', 'RISMA', 'ROMPS', 'RSMN', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'SOILSCAPE', 'STEMS', 'SWEX_POLAND', 'TAHMO', 'TERENO', 'TWENTE', 'TxSON', 'USCRN', 'WSMN', 'XMS-CAT']

=== Network: ARM ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine/ARM
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/pixel_mean/ARM
Found 16 station directories.
Loaded 16 stations after filtering, total rows: 41370
Total unique pixels: 16
Dense pixels (n_stations >= 2): 0
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 0
No dense pixels found to process.
Stations that appear in dense pixels: 0
Writing sp

## Evaluation

In [14]:
import pandas as pd
from pathlib import Path
import glob
import os

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv")

summary = pd.read_csv(SUMMARY_PATH)

# Stations that passed both filters
passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

# Stations that failed any filter
failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

dense_stations = set()
sparse_stations = set()

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    net_out = OUT_ROOT / network
    dense_dir = net_out / "dense"
    sparse_dir = net_out / "sparse"

    # collect from dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            df = pd.read_csv(f)
            if "stations" in df.columns and not df.empty:
                for st in str(df["stations"].iloc[0]).split(","):
                    st = st.strip()
                    if st:
                        dense_stations.add((network, st))

    # collect from sparse
    if sparse_dir.exists():
        for st_dir in sorted((COMBINE_ROOT / network).iterdir()):
            if not st_dir.is_dir():
                continue
            station = st_dir.name
            pattern = str(sparse_dir / f"{station}*.csv")
            files = glob.glob(pattern)
            if files:
                sparse_stations.add((network, station))

print("Total passed stations        :", len(passed_set))
print("Stations in dense pixels     :", len(dense_stations))
print("Stations in sparse outputs   :", len(sparse_stations))

print("\nStations that passed but not in dense or sparse:")
missing = passed_set - dense_stations - sparse_stations
print(missing)

print("\nStations that failed but appear in dense or sparse (should be empty):")
bad = (failed_set & dense_stations) | (failed_set & sparse_stations)
print(bad)


Total passed stations        : 1195
Stations in dense pixels     : 603
Stations in sparse outputs   : 661

Stations that passed but not in dense or sparse:
set()

Stations that failed but appear in dense or sparse (should be empty):
{('SNOTEL', 'EagleSummit'), ('NGARI', 'SQ20'), ('TAHMO', 'BimbillaSHS,Bimbilla'), ('SNOTEL', 'PortGraham'), ('HOAL', 'Hoal-08'), ('USCRN', 'Bodega-6-WSW'), ('SNOTEL', 'UpperNomeCreek'), ('SNOTEL', 'ClackamasLake'), ('NGARI', 'SQ09'), ('SNOTEL', 'MooreCreekBridge'), ('SNOTEL', 'Sunset'), ('SNOTEL', 'MunsonRidge'), ('NGARI', 'SQ16'), ('SCAN', 'GuilarteForest'), ('MySMNet', 'MPOB-BL1'), ('NGARI', 'ALI03'), ('SCAN', 'Fortuna'), ('NGARI', 'SQ10'), ('HOAL', 'Hoal-07'), ('NGARI', 'SQ11'), ('SNOTEL', 'PierceR.S.'), ('NGARI', 'SQ03'), ('SMOSMANIA', 'Prades-le-Lez'), ('SNOTEL', 'NukaGlacier'), ('NGARI', 'ALI01'), ('SNOTEL', 'SummitCreek'), ('NGARI', 'SQ21'), ('MySMNet', 'MPOB2012'), ('Ru_CFR', 'Fyodorovskoyewetsprucestand'), ('SCAN', 'SunleafNursery'), ('TAHMO', 'Wal

In [16]:
import os
import glob
from pathlib import Path
import pandas as pd

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/amsr_lprm_insitu_sd_corr_summary.csv")

# ------------------------------------------------------------------
# 1) Load summary to know which stations passed/failed
# ------------------------------------------------------------------
summary = pd.read_csv(SUMMARY_PATH)

passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

# ------------------------------------------------------------------
# 2) Scan COMBINE_ROOT: all networks / stations / files
# ------------------------------------------------------------------
combine_records = []

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    for st_dir in sorted(net_dir.iterdir()):
        if not st_dir.is_dir():
            continue
        station = st_dir.name
        files = sorted(glob.glob(str(st_dir / "*.csv")))
        for f in files:
            combine_records.append({
                "network": network,
                "station": station,
                "file_basename": os.path.basename(f),
                "source": "combine",
            })

combine_df = pd.DataFrame(combine_records)

print("COMBINE: networks =", combine_df["network"].nunique(),
      "stations =", combine_df[["network","station"]].drop_duplicates().shape[0],
      "files =", len(combine_df))

# ------------------------------------------------------------------
# 3) Scan OUT_ROOT: dense and sparse outputs, map back to stations
# ------------------------------------------------------------------
output_records = []

for net_dir in sorted(OUT_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name

    dense_dir = net_dir / "dense"
    if dense_dir.is_dir():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            try:
                df = pd.read_csv(f, usecols=["stations"])
                if df.empty:
                    stations = []
                else:
                    stations = [s.strip() for s in str(df["stations"].iloc[0]).split(",") if s.strip()]
            except Exception:
                stations = []

            if not stations:
                # still record file with unknown stations
                output_records.append({
                    "network": network,
                    "station": None,
                    "file_basename": os.path.basename(f),
                    "type": "dense",
                })
            else:
                for st in stations:
                    output_records.append({
                        "network": network,
                        "station": st,
                        "file_basename": os.path.basename(f),
                        "type": "dense",
                    })

    sparse_dir = net_dir / "sparse"
    if sparse_dir.is_dir():
        for f in glob.glob(str(sparse_dir / "*.csv")):
            # sparse file name is typically <station>*.csv
            base = os.path.basename(f)
            station = base.split("_")[0]  # adjust if naming different
            output_records.append({
                "network": network,
                "station": station,
                "file_basename": base,
                "type": "sparse",
            })

output_df = pd.DataFrame(output_records)

print("PIXEL_MEAN: networks =", output_df["network"].nunique(),
      "stations =", output_df[["network","station"]].dropna().drop_duplicates().shape[0],
      "files =", len(output_df))

# ------------------------------------------------------------------
# 4) Compare at station level
# ------------------------------------------------------------------
combine_stations = set(zip(combine_df["network"], combine_df["station"]))
output_stations  = set(zip(output_df["network"], output_df["station"]))

stations_missing_any_output = combine_stations - output_stations
stations_with_output_but_no_input = output_stations - combine_stations

print("\nStations present in COMBINE but missing in any pixel_mean output:", len(stations_missing_any_output))
print(sorted(stations_missing_any_output)[:50])  # show first 50

print("\nStations present in pixel_mean but not in COMBINE:", len(stations_with_output_but_no_input))
print(sorted(stations_with_output_but_no_input)[:50])

# ------------------------------------------------------------------
# 5) Compare at file level per station
#     - For passed stations, every input file should correspond
#       to at least one dense or sparse output file.
# ------------------------------------------------------------------
# Build quick lookup: for each (net, station) -> list of output files
out_by_station = (
    output_df
    .groupby(["network","station"])["file_basename"]
    .apply(set)
    .to_dict()
)

file_mismatches = []

for (net, st), group in combine_df.groupby(["network","station"]):
    input_files = set(group["file_basename"])
    out_files = out_by_station.get((net, st), set())

    # Only enforce matching for stations that passed filters
    status = "passed" if (net, st) in passed_set else "failed_or_unknown"

    if status == "passed":
        if not out_files:
            file_mismatches.append({
                "network": net,
                "station": st,
                "status": status,
                "missing_all_outputs": True,
                "missing_files": ";".join(sorted(input_files)),
            })
        # you could also go finer: per file mapping if there is a 1:1 naming
    else:
        # For failed stations, we might want to know if they still produced outputs
        if out_files:
            file_mismatches.append({
                "network": net,
                "station": st,
                "status": status,
                "missing_all_outputs": False,
                "missing_files": "",
            })

mismatch_df = pd.DataFrame(file_mismatches)
print("\nPassed stations with no outputs OR failed stations with outputs:", len(mismatch_df))
print(mismatch_df.head(50))

# Save detailed reports
combine_df.to_csv(OUT_ROOT / "audit_combine_listing.csv", index=False)
output_df.to_csv(OUT_ROOT / "audit_pixel_mean_listing.csv", index=False)
mismatch_df.to_csv(OUT_ROOT / "audit_mismatches.csv", index=False)


COMBINE: networks = 44 stations = 1287 files = 1287
PIXEL_MEAN: networks = 43 stations = 1263 files = 1263

Stations present in COMBINE but missing in any pixel_mean output: 24
[('FMI', 'SOD022'), ('FMI', 'SOD023'), ('HOAL', 'Hoal-19'), ('OZNET', 'Alabama'), ('OZNET', 'Dry-Lake'), ('SCAN', 'Corozal'), ('SNOTEL', 'Coldfoot'), ('SNOTEL', 'ElkButte'), ('SNOTEL', 'FortValley'), ('SNOTEL', 'JackWadeJct'), ('SNOTEL', 'LostDog'), ('SNOTEL', 'MormonMtnSummit'), ('SNOTEL', 'TogwoteePass'), ('SOILSCAPE', 'node401'), ('TAHMO', 'CoteDIvoireMeteoOffice'), ('TAHMO', 'DedanKimathiUniversityofTechnology'), ('TAHMO', 'ImurtottPrimaryschool'), ('TAHMO', 'MagomanoSecondarySchool'), ('TAHMO', 'MaraTriangleAirstrip'), ('TAHMO', 'MurungaruSecondarySchool'), ('TAHMO', 'NaboishoCamp'), ('TAHMO', 'SenchuraSecondarySchool'), ('TAHMO', 'StJohnsNyamagwa'), ('TxSON', 'CR200-11')]

Stations present in pixel_mean but not in COMBINE: 0
[]

Passed stations with no outputs OR failed stations with outputs: 68
      netw

In [12]:
import os
import glob
import shutil
import pandas as pd

COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine"
BACKUP_ROOT  = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/backup_empty_stations"

os.makedirs(BACKUP_ROOT, exist_ok=True)

empty_station_dirs = []

for network in sorted(d for d in os.listdir(COMBINE_ROOT)
                      if os.path.isdir(os.path.join(COMBINE_ROOT, d))):
    net_dir = os.path.join(COMBINE_ROOT, network)
    for station in sorted(d for d in os.listdir(net_dir)
                          if os.path.isdir(os.path.join(net_dir, d))):
        st_dir = os.path.join(net_dir, station)
        csv_files = glob.glob(os.path.join(st_dir, "*.csv"))

        # No CSV at all -> treat as empty
        if not csv_files:
            empty_station_dirs.append(st_dir)
            continue

        has_valid = False
        for f in csv_files:
            try:
                df = pd.read_csv(f, usecols=["ts_station_k", "ts_lprm"])
            except Exception as e:
                print(f"[WARN] Could not read {f}: {e}")
                continue

            mask = df["ts_station_k"].notna() & df["ts_lprm"].notna()
            if mask.any():
                has_valid = True
                break

        if not has_valid:
            empty_station_dirs.append(st_dir)

print(f"Found {len(empty_station_dirs)} empty stations.")

# Move empty stations to backup tree, preserving network/station structure
for st_dir in empty_station_dirs:
    rel = os.path.relpath(st_dir, COMBINE_ROOT)
    dest_dir = os.path.join(BACKUP_ROOT, rel)
    os.makedirs(os.path.dirname(dest_dir), exist_ok=True)
    print(f"Moving {st_dir} -> {dest_dir}")
    shutil.move(st_dir, dest_dir)

print("Done.")


Found 75 empty stations.
Moving /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine/COSMOS-UK/CocklePark -> /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/backup_empty_stations/COSMOS-UK/CocklePark
Moving /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine/COSMOS-UK/CwmGarw -> /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/backup_empty_stations/COSMOS-UK/CwmGarw
Moving /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine/COSMOS-UK/HenfaesFarm -> /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/backup_empty_stations/COSMOS-UK/HenfaesFarm
Moving /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combine/COSMOS-UK/HollinHill -> /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/backup_empty_stations/COSMOS-UK/HollinHill
Moving /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/amsr_lprm/combi